<a href="https://colab.research.google.com/github/AlperYildirim1/HAMON/blob/main/HAMON_Torchoptics_last.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q torchoptics

In [ ]:
# Clone the official ETT dataset repository
!git clone https://github.com/zhouhaoyi/ETDataset.git

# Move the ETT-small folder to the current directory to match your config paths
!mv ETDataset/ETT-small .

# Clean up the leftover repo
!rm -rf ETDataset

In [ ]:
torchoptics.set_default_spacing(10e-6)
torchoptics.set_default_wavelength(700e-9)

In [ ]:
"""
OpticalFITS v4 — Pure Diffractive Deep Neural Network (D²NN)
============================================================
A pure optical neural network for time series forecasting.
Zero digital linear layers. Zero digital mixing.
100% of the computation is performed by light diffracting through free space.

Physics:
  - Input: 336 steps of data encoded as light amplitude + 96 steps of darkness
  - 4 layers of trainable Phase Masks separated by free space (z = 3cm)
  - Light mathematically diffracts (Angular Spectrum Method), mixing the features.
  - Detector reads the light that diffracts into the 96-step dark zone.

Dataset: ETTh1
"""

import os
import math
import random
import logging
from datetime import datetime

import torch
import torch.nn as nn
import torch.optim as optim
import pandas as pd
import numpy as np
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import StandardScaler
from tqdm.auto import tqdm

# =============================================================================
# 0. CONFIGURATION
# =============================================================================

DATASETS_CONFIG = {
    "etth1": {
        "path": "ETT-small/ETTh1.csv",
        "split":[8640, 2880, 2880],
        "channels": 7,
        "batch_size": 128,
    }
}

PRED_LENS = [96]
SEQ_LEN   = 336
SEED      = 1

EPOCHS    = 50
LR        = 1e-2      # Fast LR for phase masks
PATIENCE  = 10
BF_LAMBDA = 0.5       # B+F joint supervision anchor

# --- Reproducibility ---
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark     = False

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

BASE_DIR = "."
SAVE_DIR = os.path.join(BASE_DIR, "checkpoints")
LOG_DIR  = os.path.join(BASE_DIR, "logs")
os.makedirs(SAVE_DIR, exist_ok=True)
os.makedirs(LOG_DIR, exist_ok=True)
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

def get_logger(run_name):
    logger = logging.getLogger(run_name)
    logger.setLevel(logging.INFO)
    logger.handlers.clear()
    logger.propagate = False
    fh = logging.FileHandler(os.path.join(LOG_DIR, f"{run_name}_{timestamp}.log"))
    fh.setFormatter(logging.Formatter("%(asctime)s | %(message)s", datefmt="%Y-%m-%d %H:%M:%S"))
    logger.addHandler(fh)
    ch = logging.StreamHandler()
    ch.setFormatter(logging.Formatter("%(message)s"))
    logger.addHandler(ch)
    return logger

# =============================================================================
# 1. DATASET & REVIN
# =============================================================================

class TSDataset(Dataset):
    def __init__(self, data, seq_len, pred_len):
        self.data = data
        self.seq_len = seq_len
        self.pred_len = pred_len

    def __len__(self):
        return len(self.data) - self.seq_len - self.pred_len + 1

    def __getitem__(self, idx):
        x = self.data[idx : idx + self.seq_len]
        y = self.data[idx + self.seq_len : idx + self.seq_len + self.pred_len]
        return torch.FloatTensor(x), torch.FloatTensor(y)

def load_data(dataset_name):
    cfg = DATASETS_CONFIG[dataset_name]
    df = pd.read_csv(cfg["path"])
    data = df.iloc[:, 1:].values
    train_len = cfg["split"][0]
    scaler = StandardScaler()
    scaler.fit(data[:train_len])
    data = scaler.transform(data)
    val_len = cfg["split"][1]
    return data[:train_len], data[train_len : train_len + val_len], data[train_len + val_len:], scaler, cfg

class RevIN(nn.Module):
    def __init__(self, num_features, eps=1e-5):
        super().__init__()
        self.eps = eps
        self.affine_weight = nn.Parameter(torch.ones(num_features))
        self.affine_bias   = nn.Parameter(torch.zeros(num_features))

    def forward(self, x, mode="norm"):
        if mode == "norm":
            self._mean = x.mean(dim=1, keepdim=True).detach()
            self._std  = (x.var(dim=1, keepdim=True, unbiased=False) + self.eps).sqrt().detach()
            x = (x - self._mean) / self._std
            return x * self.affine_weight + self.affine_bias
        else:
            x = (x - self.affine_bias) / self.affine_weight
            return x * self._std + self._mean

# =============================================================================
# 3. PURE DIFFRACTIVE OPTICAL NETWORK (D²NN)
# =============================================================================

class DiffractiveNet(nn.Module):
    """
    Simulates light propagating through N layers of phase masks separated by air.
    Physics engine: 1D Angular Spectrum Method (Rigorous diffraction)
    """
    def __init__(
        self,
        lookback=336,
        horizon=96,
        channels=7,
        num_layers=4,          # 4 pieces of glass
        grid_size=512,         # 512 pixels wide
        spacing=10e-6,         # 10 microns per pixel
        wavelength=1e-6,       # 1 micron wavelength (infrared)
        distance=0.03          # 3 cm of air between each mask
    ):
        super().__init__()
        self.lookback = lookback
        self.horizon = horizon
        self.channels = channels
        self.grid_size = grid_size

        # Center the data in the physical optical grid
        total_len = lookback + horizon
        self.start_idx = (grid_size - total_len) // 2

        self.revin = RevIN(channels)

        # Trainable Phase Masks (The "Weights" of the neural net)
        # N layers * 512 pixels = 2048 parameters total!
        self.phase_masks = nn.ParameterList([
            nn.Parameter(torch.zeros(grid_size)) for _ in range(num_layers)
        ])

        # --- PHYSICS PRE-COMPUTATION (Free Space Transfer Function) ---
        # How light spreads through empty space
        fx = torch.fft.fftfreq(grid_size, d=spacing)
        k = 2 * math.pi / wavelength

        # Evanescent waves decay instantly, we mask them out for stability
        evanescent_mask = (wavelength * fx)**2 <= 1.0

        # Free space propagation phase shift
        phase_shift = k * distance * torch.sqrt(torch.clamp(1.0 - (wavelength * fx)**2, min=0.0))

        # The Transfer Function H (Complex tensor)
        H = torch.exp(1j * phase_shift) * evanescent_mask
        self.register_buffer("H", H)

    def forward(self, x, return_backcast=False):
        B, L, C = x.shape

        # 1. Digital Pre-processing
        x_norm = self.revin(x, mode="norm")
        x_flat = x_norm.permute(0, 2, 1).reshape(B * C, L)

        # 2. Electro-Optical Conversion (Inject data into light beam)
        # The grid is 512 pixels. Data occupies the first 336 pixels.
        # The next 96 pixels are pure darkness (zeros).
        U = torch.zeros(B * C, self.grid_size, dtype=torch.complex64, device=x.device)
        U[:, self.start_idx : self.start_idx + L] = x_flat.to(torch.complex64)

        # 3. PURE OPTICS: Propagate through the layers
        for phase_mask in self.phase_masks:
            # Step A: Light diffracts through free space (Matrix Multiplication!)
            U_f = torch.fft.fft(U, norm='ortho')
            U_f = U_f * self.H
            U = torch.fft.ifft(U_f, norm='ortho')

            # Step B: Light passes through the learned glass mask (Activation Function)
            U = U * torch.exp(1j * phase_mask)

        # Final flight to the camera sensor
        U_f = torch.fft.fft(U, norm='ortho')
        U_f = U_f * self.H
        U = torch.fft.ifft(U_f, norm='ortho')

        # 4. Opto-Electrical Conversion (Coherent Homodyne Camera Detection)
        # Reads the Real part to allow negative normalized numbers
        amplitude = torch.real(U)

        # 5. Read the Detector
        # Read what the light reconstructed in the backcast region
        backcast_flat = amplitude[:, self.start_idx : self.start_idx + self.lookback]
        # Read the light that diffracted into the dark region (The Forecast!)
        forecast_flat = amplitude[:, self.start_idx + self.lookback : self.start_idx + self.lookback + self.horizon]

        # 6. Digital Post-processing
        backcast = backcast_flat.reshape(B, C, self.lookback).permute(0, 2, 1)
        forecast = forecast_flat.reshape(B, C, self.horizon).permute(0, 2, 1)
        forecast = self.revin(forecast, mode="denorm")

        if return_backcast:
            return forecast, backcast, x_norm
        return forecast

# =============================================================================
# 4. TRAINING LOOP
# =============================================================================

def evaluate(model, loader, criterion):
    model.eval()
    total_mse, total_mae, n = 0.0, 0.0, 0
    with torch.no_grad():
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            pred = model(x)
            total_mse += criterion(pred, y).item() * x.size(0)
            total_mae += nn.functional.l1_loss(pred, y).item() * x.size(0)
            n += x.size(0)
    return total_mse / n, total_mae / n

def train_optical_fits(dataset_name="etth1", pred_len=96):
    run_name = f"OpticalD2NN_{dataset_name}_h{pred_len}"
    logger = get_logger(run_name)
    logger.info(f"=== {run_name} ===")

    train_data, val_data, test_data, scaler, cfg = load_data(dataset_name)

    train_loader = DataLoader(TSDataset(train_data, SEQ_LEN, pred_len), batch_size=cfg["batch_size"], shuffle=True, drop_last=True)
    val_loader   = DataLoader(TSDataset(val_data, SEQ_LEN, pred_len), batch_size=cfg["batch_size"], shuffle=False)
    test_loader  = DataLoader(TSDataset(test_data, SEQ_LEN, pred_len), batch_size=cfg["batch_size"], shuffle=False)

    model = DiffractiveNet(lookback=SEQ_LEN, horizon=pred_len, channels=cfg["channels"]).to(device)

    param_count = sum(p.numel() for p in model.parameters() if p.requires_grad)
    logger.info(f"Pure Diffractive Optics | Params: {param_count:,} | Mix: Free Space")

    # High LR because phase shifts are strictly between -pi and pi, need fast adjustments
    optimizer = optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-3)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)
    criterion = nn.MSELoss()

    best_val_mse = float("inf")
    patience_counter = 0
    ckpt_path = os.path.join(SAVE_DIR, f"{run_name}.pt")

    for epoch in range(1, EPOCHS + 1):
        model.train()
        train_losses = list()
        for x, y in tqdm(train_loader, desc=f"[{run_name}] Epoch {epoch}/{EPOCHS}", leave=False):
            x, y = x.to(device), y.to(device)
            optimizer.zero_grad()

            # The light flies through the 4 glass plates
            forecast, backcast, x_norm = model(x, return_backcast=True)

            # Joint B+F Supervision (Force the optics to reconstruct AND forecast)
            loss_forecast = criterion(forecast, y)
            loss_backcast = criterion(backcast, x_norm)
            loss = loss_forecast + BF_LAMBDA * loss_backcast

            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            train_losses.append(loss_forecast.item())

        scheduler.step()
        avg_train = sum(train_losses) / len(train_losses)

        val_mse, val_mae = evaluate(model, val_loader, criterion)
        logger.info(
            f"Epoch {epoch:3d} | Train MSE: {avg_train:.6f} | "
            f"Val MSE: {val_mse:.6f} | Val MAE: {val_mae:.6f} | "
            f"LR: {scheduler.get_last_lr()[0]:.2e}"
        )

        if val_mse < best_val_mse:
            best_val_mse = val_mse
            patience_counter = 0
            torch.save(model.state_dict(), ckpt_path)
            logger.info(f"  >> Best model saved (Val MSE: {val_mse:.6f})")
        else:
            patience_counter += 1
            if patience_counter >= PATIENCE:
                logger.info(f"  >> Early stopping at epoch {epoch}")
                break

    # --- Test ---
    model.load_state_dict(torch.load(ckpt_path, weights_only=True))
    test_mse, test_mae = evaluate(model, test_loader, criterion)
    logger.info(f"TEST (Optics) | MSE: {test_mse:.6f} | MAE: {test_mae:.6f}")

    return {
        "dataset": dataset_name,
        "horizon": pred_len,
        "test_mse": test_mse,
        "test_mae": test_mae,
        "params": param_count,
    }


# =============================================================================
# 5. MAIN
# =============================================================================

if __name__ == "__main__":
    results = list()

    result = train_optical_fits("etth1", pred_len=96)
    results.append(result)

    print("\n" + "=" * 60)
    print("OPTICAL D2NN -- RESULTS SUMMARY")
    print("=" * 60)
    for r in results:
        print(f"  {r['dataset']} | H={r['horizon']:3d} | MSE={r['test_mse']:.6f} | MAE={r['test_mae']:.6f}")
    print("=" * 60)

In [ ]:
# =============================================================================
# 6c. TORCHOPTICS VALIDATION — LARGE Y GRID + PROPAGATION CHECK
# =============================================================================

import torchoptics
from torchoptics import Field, System
from torchoptics.elements import PhaseModulator, IdentityElement

torchoptics.set_default_spacing(10e-6)
torchoptics.set_default_wavelength(1e-6)

Z_GAP = 0.03
pred_len = 96

# Try larger Y grids to reduce boundary artifacts
for GRID_Y in [4, 64, 256]:
    print(f"\n{'='*60}")
    print(f"GRID_Y = {GRID_Y}  (Y aperture = {GRID_Y * 10e-6 * 1e3:.1f} mm)")
    print(f"{'='*60}")

    # Build system
    elements = []
    for i, phase_1d in enumerate(model.phase_masks):
        phase_2d = phase_1d.detach().cpu().unsqueeze(1).expand(
            model.grid_size, GRID_Y
        ).contiguous()
        elements.append(PhaseModulator(phase_2d, z=Z_GAP * (i + 1)))
    elements.append(IdentityElement((model.grid_size, GRID_Y), z=Z_GAP * 5))
    system = System(*elements)

    with torch.no_grad():
        # Same input sample
        field_1d = torch.zeros(model.grid_size, dtype=torch.complex64)
        field_1d[model.start_idx : model.start_idx + SEQ_LEN] = sample.to(
            torch.complex64
        )
        field_2d = field_1d.unsqueeze(1).expand(
            model.grid_size, GRID_Y
        ).contiguous()

        input_field = Field(field_2d)
        output_field = system(input_field)
        torchoptics_out = output_field.data[:, GRID_Y // 2].real  # Read center column

        # Scale factor
        scale = our_out.norm() / (torchoptics_out.norm() + 1e-12)

        # Pearson full grid
        our_c = our_out - our_out.mean()
        to_c = torchoptics_out - torchoptics_out.mean()
        pearson = (our_c * to_c).sum() / (our_c.norm() * to_c.norm() + 1e-12)

        # Pearson forecast zone
        forecast_zone = slice(
            model.start_idx + SEQ_LEN,
            model.start_idx + SEQ_LEN + pred_len
        )
        our_fc = our_out[forecast_zone] - our_out[forecast_zone].mean()
        to_fc = torchoptics_out[forecast_zone] - torchoptics_out[forecast_zone].mean()
        pearson_fc = (our_fc * to_fc).sum() / (our_fc.norm() * to_fc.norm() + 1e-12)

        # MAE after rescaling
        rescaled_mae = (our_out - torchoptics_out * scale).abs().mean()

        print(f"  Scale factor:          {scale.item():.2f}x")
        print(f"  Pearson (full grid):   {pearson.item():.6f}")
        print(f"  Pearson (forecast):    {pearson_fc.item():.6f}")
        print(f"  MAE after rescaling:   {rescaled_mae.item():.8f}")

        if pearson.item() > 0.99:
            print(f"  >> PASS at GRID_Y={GRID_Y}")
        elif pearson.item() > 0.95:
            print(f"  >> CLOSE at GRID_Y={GRID_Y}")
        else:
            print(f"  >> Still mismatched at GRID_Y={GRID_Y}")

# Also check what propagation TorchOptics is using
print(f"\n{'='*60}")
print("TorchOptics propagation info:")
try:
    import torchoptics.propagation as prop
    print(f"  Available: {dir(prop)}")
except Exception as e:
    print(f"  Could not inspect: {e}")

In [ ]:
# =============================================================================
# 6d. TORCHOPTICS vs OURS — ACTUAL FORECASTING MSE/MAE
# =============================================================================

import torchoptics
from torchoptics import Field, System
from torchoptics.elements import PhaseModulator, IdentityElement

torchoptics.set_default_spacing(10e-6)
torchoptics.set_default_wavelength(1e-6)

Z_GAP = 0.03
pred_len = 96
GRID_Y = 64  # Best tradeoff: good correlation, reasonable memory

# Rebuild model
model = DiffractiveNet(
    lookback=SEQ_LEN, horizon=pred_len, channels=cfg["channels"]
).to(device)
ckpt_path = os.path.join(SAVE_DIR, "OpticalD2NN_etth1_h96.pt")
model.load_state_dict(torch.load(ckpt_path, weights_only=True))
model.eval()

# Rebuild test loader
train_data, val_data, test_data, scaler, cfg = load_data("etth1")
test_loader = DataLoader(
    TSDataset(test_data, SEQ_LEN, pred_len),
    batch_size=32, shuffle=False  # Smaller batch for memory
)

# Build TorchOptics system
elements = []
for i, phase_1d in enumerate(model.phase_masks):
    phase_2d = phase_1d.detach().cpu().unsqueeze(1).expand(
        model.grid_size, GRID_Y
    ).contiguous()
    elements.append(PhaseModulator(phase_2d, z=Z_GAP * (i + 1)))
elements.append(IdentityElement((model.grid_size, GRID_Y), z=Z_GAP * 5))
system = System(*elements)

# Run full test set through BOTH engines
ours_mse_total, ours_mae_total = 0.0, 0.0
to_mse_total, to_mae_total = 0.0, 0.0
n_samples = 0

print("Running full test set through both engines...\n")

with torch.no_grad():
    for batch_idx, (x, y) in enumerate(tqdm(test_loader, desc="Testing")):
        x, y = x.to(device), y.to(device)
        B, L, C = x.shape

        # ===== OUR ENGINE (standard forward pass) =====
        forecast_ours = model(x)

        # ===== TORCHOPTICS ENGINE =====
        # Step 1: RevIN normalize (digital)
        x_norm = model.revin(x, mode="norm")
        x_flat = x_norm.permute(0, 2, 1).reshape(B * C, L)

        # Step 2: Process each sample through TorchOptics
        to_forecasts = []
        for s in range(B * C):
            # Embed in grid
            field_1d = torch.zeros(model.grid_size, dtype=torch.complex64)
            field_1d[model.start_idx : model.start_idx + L] = x_flat[s].cpu().to(
                torch.complex64
            )
            field_2d = field_1d.unsqueeze(1).expand(
                model.grid_size, GRID_Y
            ).contiguous()

            # Propagate
            input_field = Field(field_2d)
            output_field = system(input_field)

            # Read forecast zone (center Y column, real part)
            forecast_1d = output_field.data[
                model.start_idx + SEQ_LEN : model.start_idx + SEQ_LEN + pred_len,
                GRID_Y // 2
            ].real
            to_forecasts.append(forecast_1d)

        # Step 3: Reshape and denormalize
        to_forecast_flat = torch.stack(to_forecasts, dim=0)  # (B*C, pred_len)
        to_forecast = to_forecast_flat.reshape(B, C, pred_len).permute(0, 2, 1)
        to_forecast = to_forecast.to(device)
        to_forecast = model.revin(to_forecast, mode="denorm")

        # ===== METRICS =====
        ours_mse_total += nn.functional.mse_loss(forecast_ours, y).item() * B
        ours_mae_total += nn.functional.l1_loss(forecast_ours, y).item() * B
        to_mse_total += nn.functional.mse_loss(to_forecast, y).item() * B
        to_mae_total += nn.functional.l1_loss(to_forecast, y).item() * B
        n_samples += B

        if batch_idx == 0:
            print(f"  First batch sanity check:")
            print(f"    Ours MSE:       {nn.functional.mse_loss(forecast_ours, y).item():.6f}")
            print(f"    TorchOptics MSE: {nn.functional.mse_loss(to_forecast, y).item():.6f}")

ours_mse = ours_mse_total / n_samples
ours_mae = ours_mae_total / n_samples
to_mse = to_mse_total / n_samples
to_mae = to_mae_total / n_samples

print(f"\n{'='*60}")
print(f"FULL TEST SET RESULTS")
print(f"{'='*60}")
print(f"  Our 1D ASM Engine:     MSE={ours_mse:.6f}  MAE={ours_mae:.6f}")
print(f"  TorchOptics 2D Engine: MSE={to_mse:.6f}  MAE={to_mae:.6f}")
print(f"  MSE difference:        {abs(ours_mse - to_mse):.6f} ({abs(ours_mse - to_mse)/ours_mse*100:.2f}%)")
print(f"  MAE difference:        {abs(ours_mae - to_mae):.6f} ({abs(ours_mae - to_mae)/ours_mae*100:.2f}%)")
print(f"{'='*60}")

In [ ]:
# =============================================================================
# 7. ABLATION: NUMBER OF GLASS LAYERS
# =============================================================================

results_ablation = []

for num_layers in [1, 2, 4, 8, 12, 16]:
    run_name = f"HAMON_layers{num_layers}"
    logger = get_logger(run_name)

    train_data, val_data, test_data, scaler, cfg = load_data("etth1")
    train_loader = DataLoader(
        TSDataset(train_data, SEQ_LEN, 96),
        batch_size=cfg["batch_size"], shuffle=True, drop_last=True
    )
    val_loader = DataLoader(
        TSDataset(val_data, SEQ_LEN, 96),
        batch_size=cfg["batch_size"], shuffle=False
    )
    test_loader = DataLoader(
        TSDataset(test_data, SEQ_LEN, 96),
        batch_size=cfg["batch_size"], shuffle=False
    )

    model = DiffractiveNet(
        lookback=SEQ_LEN, horizon=96,
        channels=cfg["channels"],
        num_layers=num_layers
    ).to(device)

    param_count = sum(p.numel() for p in model.parameters() if p.requires_grad)
    logger.info(f"{run_name} | Params: {param_count:,}")

    optimizer = optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-3)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)
    criterion = nn.MSELoss()

    best_val_mse = float("inf")
    patience_counter = 0
    ckpt_path = os.path.join(SAVE_DIR, f"{run_name}.pt")

    for epoch in range(1, EPOCHS + 1):
        model.train()
        train_losses = []
        for x, y in tqdm(train_loader, desc=f"[{run_name}] Epoch {epoch}", leave=False):
            x, y = x.to(device), y.to(device)
            optimizer.zero_grad()
            forecast, backcast, x_norm = model(x, return_backcast=True)
            loss = criterion(forecast, y) + BF_LAMBDA * criterion(backcast, x_norm)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            train_losses.append(criterion(forecast, y).item())
        scheduler.step()

        val_mse, val_mae = evaluate(model, val_loader, criterion)

        if val_mse < best_val_mse:
            best_val_mse = val_mse
            patience_counter = 0
            torch.save(model.state_dict(), ckpt_path)
        else:
            patience_counter += 1
            if patience_counter >= PATIENCE:
                break

    model.load_state_dict(torch.load(ckpt_path, weights_only=True))
    test_mse, test_mae = evaluate(model, test_loader, criterion)

    device_length = (num_layers + 1) * 3  # cm

    results_ablation.append({
        "layers": num_layers,
        "params": param_count,
        "test_mse": test_mse,
        "test_mae": test_mae,
        "device_cm": device_length
    })

    print(f"Layers={num_layers:2d} | Params={param_count:5d} | "
          f"MSE={test_mse:.6f} | MAE={test_mae:.6f} | "
          f"Device={device_length}cm")

print(f"\n{'='*70}")
print(f"HAMON LAYER ABLATION — ETTh1 H=96")
print(f"{'='*70}")
print(f"{'Layers':>6} {'Params':>7} {'MSE':>10} {'MAE':>10} {'Device':>8}")
print(f"{'-'*70}")
for r in results_ablation:
    print(f"{r['layers']:>6d} {r['params']:>7d} {r['test_mse']:>10.6f} "
          f"{r['test_mae']:>10.6f} {r['device_cm']:>6d} cm")
print(f"{'='*70}")

In [ ]:
# Quick test: 16 layers with more epochs
EPOCHS_LONG = 100
PATIENCE_LONG = 20

model = DiffractiveNet(
    lookback=SEQ_LEN, horizon=96,
    channels=cfg["channels"],
    num_layers=16
).to(device)

optimizer = optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-3)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS_LONG)
criterion = nn.MSELoss()

best_val_mse = float("inf")
patience_counter = 0
ckpt_path = os.path.join(SAVE_DIR, "HAMON_layers16_long.pt")

for epoch in range(1, EPOCHS_LONG + 1):
    model.train()
    for x, y in tqdm(train_loader, desc=f"Epoch {epoch}", leave=False):
        x, y = x.to(device), y.to(device)
        optimizer.zero_grad()
        forecast, backcast, x_norm = model(x, return_backcast=True)
        loss = criterion(forecast, y) + BF_LAMBDA * criterion(backcast, x_norm)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
    scheduler.step()

    val_mse, val_mae = evaluate(model, val_loader, criterion)
    if epoch % 10 == 0:
        print(f"Epoch {epoch:3d} | Val MSE: {val_mse:.6f}")

    if val_mse < best_val_mse:
        best_val_mse = val_mse
        patience_counter = 0
        torch.save(model.state_dict(), ckpt_path)
    else:
        patience_counter += 1
        if patience_counter >= PATIENCE_LONG:
            print(f"Early stopping at epoch {epoch}")
            break

model.load_state_dict(torch.load(ckpt_path, weights_only=True))
test_mse, test_mae = evaluate(model, test_loader, criterion)
print(f"\n16 Layers (long training) | MSE={test_mse:.6f} | MAE={test_mae:.6f}")
print(f"Autoformer:               | MSE=0.449000")